# 02 — WOE/IV and Leakage Checks

Continues from notebook 01. Two questions here: which features are actually
driving predictions (via Weight of Evidence / Information Value, standard
in credit-risk scorecards), and whether the strongest one (`delay_days`,
which is also literally one of three components of the `bad_order` label)
is providing real signal or just letting the model reconstruct its own
label.


## WOE / IV

Fits bins on **training data only** to avoid leaking test-set information into the bin boundaries.

In [ ]:
def calculate_woe_iv(data, feature, target, bins=5, is_categorical=False):
    data = data[[feature, target]].copy()
    if is_categorical:
        data["bin"] = data[feature]
        bin_edges = None
    else:
        data["bin"], bin_edges = pd.qcut(data[feature], q=bins, duplicates="drop", retbins=True)

    grouped = data.groupby("bin", observed=True)[target].agg(["count", "sum"])
    grouped.columns = ["total", "bad"]
    grouped["good"] = grouped["total"] - grouped["bad"]

    total_good = grouped["good"].sum()
    total_bad = grouped["bad"].sum()
    grouped["pct_good"] = (grouped["good"] + 0.5) / (total_good + 0.5)
    grouped["pct_bad"] = (grouped["bad"] + 0.5) / (total_bad + 0.5)
    grouped["woe"] = np.log(grouped["pct_good"] / grouped["pct_bad"])
    grouped["iv"] = (grouped["pct_good"] - grouped["pct_bad"]) * grouped["woe"]

    iv_total = grouped["iv"].sum()
    woe_map = grouped["woe"].to_dict()
    return bin_edges, woe_map, iv_total

def apply_woe(data, feature, bin_edges, woe_map, is_categorical=False):
    if is_categorical:
        return data[feature].map(woe_map).astype(float).fillna(0)
    else:
        binned = pd.cut(data[feature], bins=bin_edges, include_lowest=True)
        return binned.map(woe_map).astype(float).fillna(0)

train_data = x_train.copy()
train_data["bad_order"] = y_train

iv_summary = {}
for feat in num_features:
    _, _, iv = calculate_woe_iv(train_data, feat, "bad_order", bins=5, is_categorical=False)
    iv_summary[feat] = iv
for feat in cat_features:
    _, _, iv = calculate_woe_iv(train_data, feat, "bad_order", is_categorical=True)
    iv_summary[feat] = iv

iv_data = pd.DataFrame.from_dict(iv_summary, orient="index", columns=["IV"]).sort_values("IV", ascending=False)
print(iv_data)


**IV rule of thumb**: <0.02 not useful, 0.02-0.1 weak, 0.1-0.3 medium, 0.3-0.5
strong, >0.5 suspiciously strong (check for leakage).

`delay_days` comes out at IV ≈ 0.39 — the "Strong" band, and by far the
highest of any feature. It's also one of three components of `bad_order`
itself (`delay_days > 7`). Worth checking whether the model is genuinely
learning from delivery delay, or partly just reconstructing its own label.


## Refit on WOE-transformed features

Replaces each raw value with its WOE value. Coefficients become directly interpretable as log-odds shifts, and bins are monotonic by construction.

In [ ]:
x_train_woe = pd.DataFrame(index=x_train.index)
x_test_woe = pd.DataFrame(index=x_test.index)

for feat in num_features:
    edges, wmap, iv = calculate_woe_iv(train_data, feat, "bad_order", bins=5, is_categorical=False)
    x_train_woe[feat + "_woe"] = apply_woe(x_train, feat, edges, wmap, is_categorical=False)
    x_test_woe[feat + "_woe"] = apply_woe(x_test, feat, edges, wmap, is_categorical=False)

for feat in cat_features:
    edges, wmap, iv = calculate_woe_iv(train_data, feat, "bad_order", is_categorical=True)
    x_train_woe[feat + "_woe"] = apply_woe(x_train, feat, edges, wmap, is_categorical=True)
    x_test_woe[feat + "_woe"] = apply_woe(x_test, feat, edges, wmap, is_categorical=True)

woe_model = LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0)
woe_model.fit(x_train_woe, y_train)
probs_woe = woe_model.predict_proba(x_test_woe)[:, 1]

print("AUC-ROC (WOE):", roc_auc_score(y_test, probs_woe))
print("AUC-PR (WOE):", average_precision_score(y_test, probs_woe))
print(classification_report(y_test, probs_woe > 0.5))


**Result**: AUC-ROC 0.683, AUC-PR 0.322 — AUC-ROC ticks up slightly over the
raw model (0.673), but AUC-PR is noticeably *lower* (vs. 0.422 raw). The raw
model stays the primary candidate; WOE is kept as an explainability
artifact, not a performance upgrade.


## Leakage sensitivity check

If `delay_days` were purely circular with the label, removing it should
collapse the model to near-random (AUC-ROC ≈ 0.5). Refitting both the raw
and WOE models without it:


In [ ]:
num_features_no_delay = [f for f in num_features if f != "delay_days"]

preprocessor_no_delay = ColumnTransformer([
    ("num", StandardScaler(), num_features_no_delay),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])
model_no_delay = Pipeline([
    ("prep", preprocessor_no_delay),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0))
])
model_no_delay.fit(x_train[num_features_no_delay + cat_features], y_train)
probs_no_delay = model_no_delay.predict_proba(x_test[num_features_no_delay + cat_features])[:, 1]

print("Raw AUC-ROC without delay_days:", roc_auc_score(y_test, probs_no_delay))
print("Raw AUC-PR without delay_days:", average_precision_score(y_test, probs_no_delay))

x_train_woe_no_delay = x_train_woe.drop(columns=["delay_days_woe"])
x_test_woe_no_delay = x_test_woe.drop(columns=["delay_days_woe"])
woe_model_no_delay = LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0)
woe_model_no_delay.fit(x_train_woe_no_delay, y_train)
probs_woe_no_delay = woe_model_no_delay.predict_proba(x_test_woe_no_delay)[:, 1]

print("WOE AUC-ROC without delay_days:", roc_auc_score(y_test, probs_woe_no_delay))
print("WOE AUC-PR without delay_days:", average_precision_score(y_test, probs_woe_no_delay))


| Model | AUC-ROC | AUC-PR |
|---|---|---|
| Raw, with delay_days | 0.673 | 0.422 |
| Raw, without delay_days | 0.593 | 0.208 |
| WOE, with delay_days | 0.683 | 0.322 |
| WOE, without delay_days | 0.614 | 0.230 |

Removing `delay_days` drops AUC-ROC to 0.593 — a real decline, but still
meaningfully above random (0.5). This means `delay_days` is doing genuine
predictive work, not simply letting the model reconstruct its own label
(a pure circularity would collapse performance to ~random). `delay_days` is
retained in the final feature set: delivery delay is an independently
well-established driver of returns in e-commerce, not an artifact of how
the label was defined — and the label has two other components
(cancellation status, review score) that don't depend on it at all.


## Conclusion

The raw model (with `delay_days`) remains the better candidate on AUC-PR.
Precision at the cost-optimal threshold is still only 0.25, though — the
leakage check ruled out one explanation (label circularity) but didn't fix
the underlying precision ceiling. Notebook 03 tries adding new features
(seller history, interaction terms) to see if there's more signal available
than this feature set captures.
